In [ ]:
#| default_exp machine_learning.ml_augmentation

> **AI-generated — pending human review.** Created by Claude Code on 2026-10-22. Review checklist: `_nbdev_development/AI_REVIEW_CHECKLIST.md` (workspace root). Remove this cell only after review passes.

Historically augmentation ran *during* vault data extraction (`08_machine_learning_35` wraps `augment_text` around the HTML pieces while building datapoints), because there was no way to augment a finished dataset afterwards. R5 separates the two: pure datasets stay untouched, and augmentation becomes a named, versioned, regenerable transform over a pure dataset via `machine_learning.ml_data.augment_dataset`.

This module is that bridge: it wraps the string mutators of `helper.latex.augment` into row-level recipes registered in `AUGMENTATION_RECIPES`. Methods are tiered — *safe* ones vary only LaTeX styling (`\mathbf` vs `\mathbb`, `$` placement) and preserve labels; *noisy* ones change content (rename Greek letters, delete words/commands/keywords) and must be chosen per task. `push_dollar_signs` is deliberately excluded: it is self-declared buggy upstream.

In [ ]:
#| export
from typing import Callable

from trouver.helper.latex.augment import (augment_text, change_font_styles_at_random,
                                          change_greek_letters_at_random, dollar_sign_manipulation,
                                          random_char_modification, random_latex_command_removal,
                                          random_word_removal, remove_font_styles_at_random,
                                          remove_math_keywords)
from trouver.machine_learning.ml_data import AUGMENTATION_RECIPES

LATEX_SAFE_METHODS: list[tuple[Callable, float]] = [
    (remove_font_styles_at_random, 0.1),
    (change_font_styles_at_random, 0.1),
    (dollar_sign_manipulation, 0.1),
]
"""Label-preserving variation: only font styling and `$` placement change."""

LATEX_NOISY_METHODS: list[tuple[Callable, float]] = LATEX_SAFE_METHODS + [
    (change_greek_letters_at_random, 0.05),
    (random_char_modification, 0.02),
    (random_word_removal, 0.03),
    (random_latex_command_removal, 0.03),
    (remove_math_keywords, 0.1),
]
"""Safe methods plus content-changing noise; pick per task — keyword/word removal can invalidate labels."""

In [ ]:
#| export
def make_latex_recipe(column: str, methods: list[tuple[Callable, float]]) -> Callable:
    """
    Build an `ml_data` row recipe that applies all `methods` to `row[column]`.

    All other columns (labels, provenance) pass through untouched; every
    random choice comes from the recipe's own `rng`, so the augmented
    dataset is regenerable from (pure dataset, recipe id, seed).

    **Parameters**
    - `column` (*str*) — The row column holding the LaTeX text to mutate.
    - `methods` (*list[tuple[Callable, float]]*) — `(method, p)` pairs, e.g. `LATEX_SAFE_METHODS`.
    """
    def recipe(row: dict, rng) -> dict:
        funcs = [lambda x, m=m, p=p: m(x, p=p, rng=rng) for m, p in methods]
        row[column] = augment_text(row[column], funcs)
        return row
    return recipe


def register_latex_recipe(name: str, column: str,
                          methods: list[tuple[Callable, float]], version: int = 1) -> str:
    """
    Build a LaTeX recipe with `make_latex_recipe` and register it as `<name>_v<version>`.

    **Parameters**
    - `name` (*str*) — Recipe name, e.g. `'latex_robust'`.
    - `column` (*str*) — Row column to mutate.
    - `methods` (*list[tuple[Callable, float]]*) — `(method, p)` pairs.
    - `version` (*int*) — Bump when the method list or probabilities change.
    """
    recipe_id = f'{name}_v{version}'
    AUGMENTATION_RECIPES[recipe_id] = make_latex_recipe(column, methods)
    return recipe_id

In [ ]:
#| export
# Standard recipes for datasets whose text lives in a `text` column; tasks
# with other columns register their own via `register_latex_recipe`.
register_latex_recipe('latex_robust', 'text', LATEX_SAFE_METHODS)
register_latex_recipe('latex_noise', 'text', LATEX_NOISY_METHODS)

In [ ]:
import tempfile
from pathlib import Path

from fastcore.test import *

from trouver.machine_learning import ml_data as mld

In [ ]:
# The standard recipes are registered and flow through `augment_dataset`.
assert 'latex_robust_v1' in AUGMENTATION_RECIPES and 'latex_noise_v1' in AUGMENTATION_RECIPES

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    schema = mld.DatasetSchema('demo', 1, ('Note name', 'text'))
    pure = [{'Note name': 'a', 'text': r'$\mathbf{x} + \alpha$ is dense.'},
            {'Note name': 'b', 'text': r'Let $V$ be $\mathrm{span}(S)$. Definition 2.1.1 holds.'}]
    mld.save_rows(pure, mld.dataset_dir(root, schema) / 'demo.jsonl')

    out1 = mld.augment_dataset(root, schema, 'latex_robust_v1', seed=5)
    out2 = mld.augment_dataset(root, schema, 'latex_robust_v1', seed=5)
    test_eq(out1, out2)  # Regenerable through the real LaTeX mutators.
    test_eq([r['Note name'] for r in out1], ['a', 'b'])
    test_eq([r[mld.AUGMENTATION_COLUMN] for r in out1], ['latex_robust_v1'] * 2)

    # Noisy recipe runs end to end without crashing on mixed content.
    mld.augment_dataset(root, schema, 'latex_noise_v1', seed=5)

In [ ]:
# A forced-p recipe actually mutates the text column and nothing else.
register_latex_recipe('force_bold_removal', 'text', [(remove_font_styles_at_random, 1.0)])
with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    schema = mld.DatasetSchema('demo', 1, ('Note name', 'text'))
    mld.save_rows([{'Note name': 'a', 'text': r'$\mathbf{x}$'}],
                  mld.dataset_dir(root, schema) / 'demo.jsonl')
    out = mld.augment_dataset(root, schema, 'force_bold_removal_v1', seed=0)
    test_eq(out[0]['text'], '$x$')

# `push_dollar_signs` is excluded from every standard recipe (upstream-buggy).
import trouver.helper.latex.augment as aug
for tier in (LATEX_SAFE_METHODS, LATEX_NOISY_METHODS):
    assert not any(m is aug.push_dollar_signs for m, _ in tier)

In [ ]:
#| hide
# Recipes must not read global random state: identical output under a
# perturbed global stream proves the row `rng` is the only source.
import random as _random
_recipe = make_latex_recipe('text', LATEX_SAFE_METHODS)
_row = {'Note name': 'a', 'text': r'$\mathbf{x} + \mathbb{y}$'}
_random.seed(12345)
a = _recipe(dict(_row), _random.Random(7))
_random.seed(999)
b = _recipe(dict(_row), _random.Random(7))
test_eq(a, b)

## Newline-stripping for `note_division`

Division rows join the left and right notes with `\n`, so in the canonical text a newline token sits at *every* positive seam — an input-visible cue that is absent at inference on an undivided note, and intra-note line wraps make newlines anti-correlated with seams besides. Replacing every `\n` with a space removes the cue from training; applying the same transform to notes at inference keeps train and test views identical. The replacement is one-for-one, so character offsets (the zero-width seam spans) stay valid untouched — no span remapping needed.

In [ ]:
#| export
def strip_newlines_text(text: str) -> str:
    """
    Replace every newline in `text` with a space — the division view transform.

    Single source of truth for train *and* inference: training rows pass through
    it via `make_strip_newlines_recipe`; deployment must apply this exact function
    to note text before prediction so the model sees the view it trained on
    (the `\n` seam cue must be absent on both sides). Length-preserving, so
    zero-width span offsets stay valid untouched.

    **Parameters**
    - `text` (*str*) — Text to flatten.
    """
    return text.replace('\n', ' ')


def make_strip_newlines_recipe(column: str = 'text') -> Callable:
    """
    Build an `ml_data` row recipe replacing every newline in `row[column]` with a space.

    Length-preserving, so `spans` character offsets stay valid without remapping;
    deterministic (the `rng` is ignored), so the augmented dataset is trivially
    regenerable. Designed for `note_division` rows, whose `\n` joiner would
    otherwise mark every positive seam in the model input (see notebook text);
    apply `strip_newlines_text` to note text at inference so views match.

    **Parameters**
    - `column` (*str*) — The row column holding the text to flatten.
    """
    def recipe(row: dict, rng) -> dict:
        row[column] = strip_newlines_text(row[column])
        return row
    return recipe


def register_strip_newlines_recipe(name: str, column: str = 'text',
                                     version: int = 1) -> str:
    """
    Build a newline-strip recipe with `make_strip_newlines_recipe` and register it as `<name>_v<version>`.

    **Parameters**
    - `name` (*str*) — Recipe name, e.g. `'division_strip_newlines'`.
    - `column` (*str*) — Row column to flatten.
    - `version` (*int*) — Bump when the transform changes.
    """
    recipe_id = f'{name}_v{version}'
    AUGMENTATION_RECIPES[recipe_id] = make_strip_newlines_recipe(column)
    return recipe_id


In [ ]:
#| export
# The division task trains on the flattened view; eval stays pure (rule R5).
register_strip_newlines_recipe('division_strip_newlines')

In [ ]:
# Newline strip: length-preserving, spans stay valid, other keys untouched.
test_eq(strip_newlines_text('a\nb\nc'), 'a b c')   # the shared view transform itself
recipe = make_strip_newlines_recipe()
row = {'task': 'note_division', 'Note name': '_index_x',
       'text': 'ab\ncd\nef', 'spans': [{'start': 2, 'end': 2, 'label': 'split'}]}
out = recipe(dict(row), None)
test_eq(out['text'], 'ab cd ef')
test_eq(out['text'], strip_newlines_text(row['text']))  # recipe delegates to the shared transform
test_eq(len(out['text']), len(row['text']))
test_eq(out['spans'], row['spans'])  # offsets unchanged, seam char still non-blank-left
test_eq(out['Note name'], '_index_x')


In [ ]:
# End-to-end through augment_dataset on real note_division schema rows.
from trouver.machine_learning.producers.note_division import (NOTE_DIVISION_SCHEMA,
                                                                save_note_division_dataset)
with tempfile.TemporaryDirectory() as td:
    root = Path(td)
    save_note_division_dataset(
        [{'task': 'note_division', 'Note name': '_index_x',
          'text': 'first note.\nsecond note.',
          'spans': [{'start': 12, 'end': 12, 'label': 'split'}]}], root)
    aug = mld.augment_dataset(root, NOTE_DIVISION_SCHEMA, 'division_strip_newlines_v1')
    test_eq(len(aug), 1)
    assert '\n' not in aug[0]['text']
    test_eq(aug[0]['text'], 'first note. second note.')
    test_eq(aug[0]['spans'][0]['start'], 12)
    test_eq(aug[0][mld.AUGMENTATION_COLUMN], 'division_strip_newlines_v1')
    test_eq(aug[0][mld.PROVENANCE_COLUMN], '_index_x')


## Span-aware LaTeX noise (offset-span tasks)

`make_latex_recipe` rewrites the whole text, so length-changing edits shift
every later character offset — it silently corrupts `spans` on tasks whose
labels are offsets (`note_division`, def/notat NER). The fix is per-edit
bookkeeping: apply each method separately, align old↔new text with `difflib`,
and remap the spans through the alignment. A seam that lies *inside* replaced
or deleted text has no faithful counterpart, so only that one method is
skipped for the row — the rest of the noise still applies and almost no rows
are lost.

The registered division recipes apply latex noise to the pure text first, then
the newline strip last, so the final view matches `division_strip_newlines_v1`
(the baseline) plus noise; the frozen eval set on disk stays pure (R5).

In [ ]:
#| export
import difflib


def remap_spans(old_text: str, new_text: str, spans: list[dict]) -> list[dict] | None:
    """
    Map zero-width span offsets through a text edit via `difflib` alignment.

    A seam survives only where its old position lies in (or on the border of)
    an unchanged (`equal`) region; a seam strictly inside replaced or deleted
    text has no faithful counterpart, so `None` is returned and the caller
    drops that edit rather than silently mislabeling. Border positions are
    unambiguous: the two `equal` blocks flanking one edit agree on the new
    offset (`j2 == j1` for consecutive opcodes).

    **Parameters**
    - `old_text` (*str*) — Text before the edit.
    - `new_text` (*str*) — Text after the edit.
    - `spans` (*list[dict]*) — Zero-width spans in old-text coordinates.
    """
    if old_text == new_text:
        return [dict(s) for s in spans]
    dest = [-1] * (len(old_text) + 1)
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(None, old_text, new_text).get_opcodes():
        if tag == 'equal':
            for k in range(i1, i2 + 1):
                dest[k] = j1 + (k - i1)
    out = []
    for s in spans:
        if dest[s['start']] < 0:
            return None
        t = dict(s); t['start'] = t['end'] = dest[s['start']]; out.append(t)
    return out


def _apply_methods(text: str, spans: list[dict], methods: list[tuple[Callable, float]],
                   rng) -> tuple[str, list[dict]]:
    """Edit `text` with each `(method, p)` in turn, remapping `spans`; skip seam-destroyers."""
    for m, p in methods:
        new = m(text, p=p, rng=rng)
        remapped = remap_spans(text, new, spans)
        if remapped is not None:
            text, spans = new, remapped
    return text, spans


def make_span_aware_latex_recipe(column: str, methods: list[tuple[Callable, float]],
                                 spans_column: str = 'spans') -> Callable:
    """
    LaTeX-noise recipe that keeps `row[spans_column]` offsets aligned with the text.

    Plain `make_latex_recipe` cannot serve offset-span tasks: length-changing
    edits shift every later offset. Here each method is applied separately and
    its edit tracked through `remap_spans`; a method that would land on a seam
    is skipped *for that row only*, so the rest of the noise still applies and
    almost no rows are lost.

    **Parameters**
    - `column` (*str*) — Row column holding the LaTeX text.
    - `methods` (*list[tuple[Callable, float]]*) — `(method, p)` pairs, e.g. `LATEX_SAFE_METHODS`.
    - `spans_column` (*str*) — Row column holding zero-width spans to remap.
    """
    def recipe(row: dict, rng) -> dict:
        row[column], row[spans_column] = _apply_methods(
            row[column], row[spans_column], methods, rng)
        return row
    return recipe

In [ ]:
#| export
def compose_recipes(*recipes: Callable) -> Callable:
    """
    Apply row recipes left to right; a `None` from any of them drops the row.

    **Parameters**
    - `recipes` (*Callable*) — Row recipes, each `fn(row, rng) -> dict | None`.
    """
    def recipe(row: dict, rng) -> dict:
        for r in recipes:
            row = r(row, rng)
            if row is None:
                return None
        return row
    return recipe


def register_span_aware_latex_recipe(name: str, column: str = 'text',
                                     methods: list[tuple[Callable, float]] = LATEX_SAFE_METHODS,
                                     version: int = 1) -> str:
    """
    Build a span-aware LaTeX recipe with `make_span_aware_latex_recipe` and register it as `<name>_v<version>`.

    **Parameters**
    - `name` (*str*) — Recipe name, e.g. `'division_latex_robust'`.
    - `column` (*str*) — Row column to mutate.
    - `methods` (*list[tuple[Callable, float]]*) — `(method, p)` pairs.
    - `version` (*int*) — Bump when the method list or probabilities change.
    """
    recipe_id = f'{name}_v{version}'
    AUGMENTATION_RECIPES[recipe_id] = make_span_aware_latex_recipe(column, methods)
    return recipe_id

In [ ]:
#| export
# Division rows train on the stripped view with latex noise applied first (on pure
# text, seam offsets remapped per method); eval stays pure (rule R5).
AUGMENTATION_RECIPES['division_latex_robust_v1'] = compose_recipes(
    make_span_aware_latex_recipe('text', LATEX_SAFE_METHODS), make_strip_newlines_recipe())
AUGMENTATION_RECIPES['division_latex_noise_v1'] = compose_recipes(
    make_span_aware_latex_recipe('text', LATEX_NOISY_METHODS), make_strip_newlines_recipe())

In [ ]:
# remap_spans: identity, shifts through insertions, drop when a seam is destroyed.
sp = [{'start': 5, 'end': 5, 'label': 'split'}]
test_eq(remap_spans('alpha beta', 'alpha beta', sp), sp)
test_eq(remap_spans('alpha beta', 'ZZ alpha beta', sp)[0]['start'], 8)  # insert-before shifts by the inserted width
test_eq(remap_spans('alpha beta', 'alpha  beta', sp)[0]['start'], 5)   # insert at seam border: stays
assert remap_spans('aXXbYc', 'abc', [{'start': 2, 'end': 2}]) is None  # seam inside deleted text

In [ ]:
import random

# Span-aware recipe: insertion shifts the seam; a seam-destroying method is skipped.
row = {'text': 'alpha beta', 'spans': [dict(s) for s in sp]}
rec = make_span_aware_latex_recipe('text', [(lambda x, p=1.0, rng=None: 'Z' + x, 1.0)])
out = rec(dict(row), random.Random(0))
test_eq(out['text'], 'Zalpha beta'); test_eq(out['spans'][0]['start'], 6)
wipe = make_span_aware_latex_recipe('text', [(lambda x, p=1.0, rng=None: 'x' * len(x), 1.0)])
test_eq(wipe(dict(row), random.Random(0)), row)

# Composed with the newline strip: noise first, flattened view last.
comp = compose_recipes(make_span_aware_latex_recipe('text', [(lambda x, p=1.0, rng=None: 'Z' + x, 1.0)]),
                       make_strip_newlines_recipe())
out2 = comp({'text': 'alpha\nbeta', 'spans': [{'start': 5, 'end': 5}]}, random.Random(0))
test_eq(out2['text'], 'Zalpha beta'); test_eq(out2['spans'][0]['start'], 6)

# Registered division recipes: regenerable from (row, seed), seam survives, stripped view.
for rid in ('division_latex_robust_v1', 'division_latex_noise_v1'):
    f = AUGMENTATION_RECIPES[rid]
    src = {'text': 'Let $x$ be $\\alpha$. Note\nMore text.', 'spans': [{'start': 20, 'end': 20}]}
    a, b = f(dict(src), random.Random(3)), f(dict(src), random.Random(3))
    test_eq(a, b)
    test_eq(len(a['spans']), 1)
    assert '\n' not in a['text']

## Intensity-tiered augmentation for `note_division` (v2 recipe)

The v1 recipes apply uniform probabilities to every datapoint. `division_aug_v2`
draws one **intensity tier per datapoint** from the row rng — low/medium/high
scale all method probabilities by 0.5/1/2 with weights 40/35/25 (mass sits at
the cheap end: light edits are almost never harmful, heavy ones can gut the
context around a seam even when `remap_spans` keeps the position honest) — and
uses a division-tuned method set. Because the label is a *position*, it survives
almost any edit except one landing across the seam (handled by the per-method
skip), so character modification and Greek renaming join the base tier, while
word/command deletion — the only methods that can remove seam context outright
— appear in the high tier alone. Strip-newlines stays full normalization
(train/inference view identity), applied last.

In [ ]:
#| export
def make_tiered_latex_recipe(column: str, base_methods: list[tuple[Callable, float]],
                             high_only_methods: list[tuple[Callable, float]] = (),
                             tier_weights: tuple = ((0.5, 0.40), (1.0, 0.35), (2.0, 0.25)),
                             spans_column: str = 'spans') -> Callable:
    """
    Span-aware recipe whose edit probabilities are scaled by a per-row intensity tier.

    Each row draws a multiplier from `tier_weights` (`(multiplier, weight)` pairs)
    using the row rng; base methods run at p*multiplier and `high_only_methods`
    join only when the multiplier exceeds 1. Edits apply through
    `make_span_aware_latex_recipe`'s seam-safe machinery, so labels stay honest
    at every intensity, and the whole recipe stays regenerable from (row, seed).

    **Parameters**
    - `column` (*str*) — Row column holding the LaTeX text.
    - `base_methods` (*list[tuple[Callable, float]]*) — Methods active at every tier.
    - `high_only_methods` (*list[tuple[Callable, float]]*) — Methods only at tiers with multiplier > 1.
    - `tier_weights` (*tuple*) — `(multiplier, weight)` lottery; weights need not sum to 1.
    - `spans_column` (*str*) — Row column holding zero-width spans to remap.
    """
    mults = [m for m, _ in tier_weights]
    weights = [w for _, w in tier_weights]

    def recipe(row: dict, rng) -> dict:
        mult = rng.choices(mults, weights=weights)[0]
        methods = [(m, p * mult) for m, p in base_methods]
        if mult > 1.0:
            methods += [(m, p * mult) for m, p in high_only_methods]
        row[column], row[spans_column] = _apply_methods(
            row[column], row[spans_column], methods, rng)
        return row
    return recipe

In [ ]:
#| export
# Division-tuned v2: char/Greek noise in the base tier (zero-width position labels
# survive them); word/command deletion — the only seam-context removers — high tier only.
DIVISION_V2_METHODS: list[tuple[Callable, float]] = [
    (change_font_styles_at_random, 0.1),
    (remove_font_styles_at_random, 0.1),
    (dollar_sign_manipulation, 0.1),
    (random_char_modification, 0.05),
    (change_greek_letters_at_random, 0.05),
]
DIVISION_V2_HIGH_ONLY: list[tuple[Callable, float]] = [
    (random_word_removal, 0.03),
    (random_latex_command_removal, 0.03),
]
AUGMENTATION_RECIPES['division_aug_v2'] = compose_recipes(
    make_tiered_latex_recipe('text', DIVISION_V2_METHODS, DIVISION_V2_HIGH_ONLY),
    make_strip_newlines_recipe())

In [ ]:
import random

# Tiered recipe: deterministic per (row, seed); seam survives; stripped view.
for _tw in (((0.5, 1.0),), ((1.0, 1.0),), ((2.0, 1.0),)):   # force each tier alone
    rec = make_tiered_latex_recipe('text', DIVISION_V2_METHODS, DIVISION_V2_HIGH_ONLY, _tw)
    r0 = {'text': 'Let $\\alpha$ be $x$. Note\nMore words here.',
          'spans': [{'start': 20, 'end': 20}]}
    a, b = rec(dict(r0), random.Random(7)), rec(dict(r0), random.Random(7))
    test_eq(a, b)
    assert len(a['spans']) == 1 and '\n' not in compose_recipes(rec, make_strip_newlines_recipe())(dict(r0), random.Random(7))['text']

# Tier lottery actually varies rows: with equal weights over far-apart multipliers,
# two seeds produce different texts on a long-enough row.
rec = make_tiered_latex_recipe('text', [(lambda x, p=1.0, rng=None: x + 'z' * int(10 * p), 0.5)],
                               tier_weights=((0.2, 0.5), (2.0, 0.5)))
row = {'text': 'alpha beta', 'spans': [{'start': 5, 'end': 5}]}
outs = {rec(dict(row), random.Random(s))['text'] for s in range(30)}
assert len(outs) > 1, 'tier lottery must vary the augmented population'

# Registered v2 flows through augment_dataset end to end (schema-valid rows).
with tempfile.TemporaryDirectory() as td:
    root = Path(td)
    save_note_division_dataset(
        [{'task': 'note_division', 'Note name': '_index_x',
          'text': 'first note $\\mathbf{x}$ here.\nsecond note.',
          'spans': [{'start': 26, 'end': 26, 'label': 'split'}]}], root)
    aug = mld.augment_dataset(root, NOTE_DIVISION_SCHEMA, 'division_aug_v2')
    assert '\n' not in aug[0]['text'] and len(aug[0]['spans']) == 1
    test_eq(aug[0][mld.AUGMENTATION_COLUMN], 'division_aug_v2')